In [182]:
import pandas as pd
from datetime import timezone, datetime, timedelta
from storage.ib import load_many

import plotly.graph_objects as go
import plotly.subplots as sub
from jup.helpers.charts import (
    get_layout,
    get_price_trace,
    add_trade_markers,
    remove_x_gaps,
    detect_range_rule,
)
from jup.helpers.data_sources import get_exante_data, resample_ohlc


pd.options.display.width = 130



In [216]:
BID_ASK_COLUMNS_MAP = {
    "open": "av_bid",
    "high": "max_ask",
    "low": "min_bid",
    "close": "av_ask",
}


start_dt = datetime(2021, 7, 1, tzinfo=timezone.utc).date()

# df = load_many(["COPX.ARCA", "URA.ARCA"], ["TRADES", "BIDASK"], start=start_dt)

df_copx = load_many(["COPX.ARCA"], ["TRADES"], start=start_dt)[["average", "rth"]]
df_copx = df_copx[df_copx["rth"] == "1"]
df_copx = df_copx.rename(columns={"average": "copx"})

df_hg = load_many(["HG.NYMEX"], ["TRADES"], start=start_dt)[["average", "volume"]]
df_hg = df_hg[df_hg["volume"] != 0]
df_hg = df_hg.rename(columns={"average": "hg"})

df_es = load_many(["ES.GLOBEX"], ["MIDPOINT"], start=start_dt)[["close"]]
df_es = df_es.rename(columns={"close": "es"})
df_es["es"] = df_es["es"].astype(float)

df = pd.concat([df_copx, df_es, df_hg], axis=1)
df.sort_index(inplace=True)
df = df.resample("1H").mean()

# print(df)
# print(df_hg)
# print(pd.concat([df_copx, df_es, df_hg], axis=1))

# df.dropna(inplace=True)
df = df[(df['hg'].notna()) | (df['copx'].notna()) | (df['es'].notna())]


df["hg_cor"] = df["hg"] * 8.2
df["es_cor"] = (df["es"] * df["es"]) * df["hg"] / 2250000 + 1

# print(df.head(830).tail(50))

# df = df.head(13000)

layout = get_layout()

fig = sub.make_subplots(
    figure=go.Figure(layout=layout),
    rows=5,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.02,
    row_heights=[0.6, 0.1, 0.1, 0.1, 0.1],
)
# Правильный rangeslider
fig.add_trace(go.Scatter(x=df.index, y=df["hg"]), row=3, col=1)
fig.update_layout(xaxis3=dict(rangeslider=dict(visible=True, thickness=0.05)))
fig['layout']['xaxis2']['visible'] = False
fig['layout']['yaxis3']['visible'] = False


# Индикаторы на графике
line_1 = dict(color="red", width=2)
line_2 = dict(color="blue", width=1)
line_3 = dict(color="green", width=1)
fig.add_trace(go.Scatter(x=df.index, y=df["copx"], line=line_1, name="copx", marker=dict(size=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=df["hg_cor"], line=line_2, name="hg", marker=dict(size=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=df["es_cor"], line=line_3, name="es", marker=dict(size=1.5)))


index_start = df.index[0]
index_end = df.index[-1]

close_na = df["hg"].asfreq("1H").isna()
gaps = close_na[close_na.values].index
data_rangebreaks = {
    "values": list(gaps),
    "dvalue": 60 * 60 * 1000,
}
fig.update_xaxes(
    range=[index_start, index_end],
    rangebreaks=[data_rangebreaks],
)

fig.show()
